# NB03 — Retrieval Indexing
**Purpose:** Build all retrieval indexes needed for ablation (NB06) and Δgap measurement (NB07).

**Two corpora, two purposes:**
- `hotpotqa_source_docs.jsonl` → BM25 + BGE-M3 indexes for ablation (NB06), dataset-native
- `wikipedia_id_corpus.jsonl` → BM25 + BGE-M3 indexes for Δgap (NB07) and RAGAS (NB08), open-domain

**Outputs → `crosslingual-rag-indexes` Kaggle Dataset:**
```
indexes/bm25_hotpotqa.pkl
indexes/bm25_wikipedia_id.pkl
indexes/faiss_bge_m3_hotpotqa/
indexes/faiss_bge_m3_wikipedia_id/
results/retrieval_sanity_check.json
```

**Est. GPU time:** ~2–3 hours on T4 (dominated by Wikipedia ID BGE-M3 embedding).

**Strategy:** Build per-corpus, checkpoint immediately after each corpus. Do NOT wait for all indexes to finish before persisting.

## 0. Install dependencies

In [ ]:
!pip install -q rank_bm25 PySastrawi FlagEmbedding faiss-cpu huggingface_hub

## 1. Imports & Setup

In [ ]:
import os
import json
import pickle
import shutil
import time
import numpy as np
import faiss
from pathlib import Path
from tqdm import tqdm

# BM25 + Sastrawi
from rank_bm25 import BM25Okapi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# BGE-M3
from FlagEmbedding import FlagModel, BGEM3FlagModel

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Paths & Directories

In [ ]:
# ── Input paths (from crosslingual-rag-data dataset) ──────────────────────────
HOTPOTQA_DOCS_PATH  = "/kaggle/input/datasets/melisaolivia/crosslingual-rag-data/hotpotqa_source_docs.jsonl"
WIKIPEDIA_ID_PATH   = "/kaggle/input/datasets/melisaolivia/crosslingual-rag-data/wikipedia_id_corpus.jsonl"
XQUAD_PARALLEL_PATH = "/kaggle/input/datasets/melisaolivia/crosslingual-rag-data/xquad_id_parallel.jsonl"

# ── Output directories ─────────────────────────────────────────────────────────
WORKING_DIR   = Path("/kaggle/working")
INDEX_DIR     = WORKING_DIR / "indexes"
RESULTS_DIR   = WORKING_DIR / "results"
FAISS_HOTPOT  = INDEX_DIR / "faiss_bge_m3_hotpotqa"
FAISS_WIKI    = INDEX_DIR / "faiss_bge_m3_wikipedia_id"

for d in [INDEX_DIR, RESULTS_DIR, FAISS_HOTPOT, FAISS_WIKI]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready:")
for d in [INDEX_DIR, RESULTS_DIR, FAISS_HOTPOT, FAISS_WIKI]:
    print(f"  {d}")

## 3. Helper: Load JSONL

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_text(record):
    """
    Extract a single text string from a document record.
    Handles different JSONL schemas from NB01:
      - {"text": ...}
      - {"content": ...}
      - {"title": ..., "text": ...} → "title. text"
    Returns empty string if no usable field found.
    """
    title = record.get("title", "").strip()
    text  = record.get("text", record.get("content", "")).strip()
    if title and text:
        return f"{title}. {text}"
    return text or title

# Quick check on file accessibility
for label, path in [("HotpotQA docs", HOTPOTQA_DOCS_PATH),
                     ("Wikipedia ID",  WIKIPEDIA_ID_PATH),
                     ("XQuAD parallel",XQUAD_PARALLEL_PATH)]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{label}: {'✓' if exists else '✗ MISSING'}  ({size:.1f} MB)")

## 4. Load Corpora

In [ ]:
print("Loading HotpotQA source docs...")
hotpotqa_docs = load_jsonl(HOTPOTQA_DOCS_PATH)
hotpotqa_texts = [extract_text(d) for d in hotpotqa_docs]
hotpotqa_ids   = [d.get("id", str(i)) for i, d in enumerate(hotpotqa_docs)]
print(f"  Loaded {len(hotpotqa_texts)} HotpotQA docs")
print(f"  Sample: {hotpotqa_texts[0][:120]}...")

print()
print("Loading Wikipedia ID corpus...")
wiki_docs  = load_jsonl(WIKIPEDIA_ID_PATH)
wiki_texts = [extract_text(d) for d in wiki_docs]
wiki_ids   = [d.get("id", str(i)) for i, d in enumerate(wiki_docs)]
print(f"  Loaded {len(wiki_texts)} Wikipedia ID docs")
print(f"  Sample: {wiki_texts[0][:120]}...")

# Sanity: warn if any text is empty
empty_hotpot = sum(1 for t in hotpotqa_texts if not t.strip())
empty_wiki   = sum(1 for t in wiki_texts if not t.strip())
if empty_hotpot > 0:
    print(f"  ⚠ {empty_hotpot} empty HotpotQA texts — check extract_text()")
if empty_wiki > 0:
    print(f"  ⚠ {empty_wiki} empty Wikipedia ID texts — check extract_text()")

## 5. Sastrawi Preprocessor (for BM25)

In [ ]:
# Sastrawi for Indonesian-aware stemming + stopword removal
# Used for BM25 tokenization of BOTH corpora.
# HotpotQA docs are English — Sastrawi will be suboptimal but is kept
# for consistency with the baseline paper's approach.
# The asymmetry this creates (Sastrawi degrades EN) is empirically demonstrated
# in NB07 Δgap measurement.

stemmer_factory  = StemmerFactory()
stopword_factory = StopWordRemoverFactory()
stemmer   = stemmer_factory.create_stemmer()
stopwords = set(stopword_factory.get_stop_words())

def sastrawi_tokenize(text):
    """Tokenize text using Sastrawi stemmer + stopword removal."""
    text = text.lower()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Quick test
test = "Universitas Indonesia adalah universitas terbaik di Jakarta"
print(f"Input:    {test}")
print(f"Tokens:   {sastrawi_tokenize(test)}")

## 6. Build BM25 Indexes

In [ ]:
def build_bm25_index(texts, ids, corpus_name, out_path):
    """
    Tokenize corpus and build BM25Okapi index.
    Saves both the BM25 object and doc_ids to a single pickle.
    """
    print(f"[BM25] Tokenizing {corpus_name} ({len(texts)} docs)...")
    t0 = time.time()
    tokenized = [sastrawi_tokenize(t) for t in tqdm(texts, desc="Tokenizing")]
    print(f"  Tokenization done in {time.time()-t0:.1f}s")

    print(f"[BM25] Building index...")
    t0 = time.time()
    bm25 = BM25Okapi(tokenized)
    print(f"  Index built in {time.time()-t0:.1f}s")

    payload = {"bm25": bm25, "doc_ids": ids, "texts": texts}
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"  Saved to {out_path} ({size_mb:.1f} MB)")
    return bm25

bm25_hotpotqa = build_bm25_index(
    hotpotqa_texts, hotpotqa_ids,
    "HotpotQA source docs",
    INDEX_DIR / "bm25_hotpotqa.pkl"
)

print()
bm25_wikipedia = build_bm25_index(
    wiki_texts, wiki_ids,
    "Wikipedia ID corpus",
    INDEX_DIR / "bm25_wikipedia_id.pkl"
)

## 7. Build BGE-M3 FAISS Indexes

BGE-M3 supports three retrieval modes:
- **Dense**: standard embedding similarity — cross-lingual, semantic
- **Sparse**: learned sparse weights — cross-lingual BM25 equivalent (ColBERT mode skipped — expensive, not used in pipeline)

Both modes use `BGEM3FlagModel`. We store dense embeddings in FAISS (cosine via inner product on normalized vectors) and sparse weights in a separate dict for RRF fusion at query time.

**Memory note:** BGE-M3 fp16 uses ~2GB VRAM. Batch size 32 is safe on T4.

In [ ]:
print("Loading BGEM3FlagModel (fp16)...")
bge_model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print(f"Model loaded. Device: {next(bge_model.model.parameters()).device}")

In [ ]:
def embed_corpus_bge_m3(texts, batch_size=32, desc="Embedding"):
    """
    Embed corpus with BGE-M3. Returns:
      - dense_embeddings: np.ndarray of shape (N, 1024), float32, L2-normalized
      - sparse_weights:   list of dicts {token_id: weight} per document
    """
    all_dense   = []
    all_sparse  = []

    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i : i + batch_size]
        output = bge_model.encode(
            batch,
            batch_size=batch_size,
            max_length=512,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False  # ColBERT skipped — expensive, not in pipeline
        )
        dense  = output['dense_vecs']           # (batch, 1024) float32
        sparse = output['lexical_weights']      # list of dicts

        # L2 normalize dense for cosine similarity via inner product
        norms = np.linalg.norm(dense, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        dense = dense / norms

        all_dense.extend(dense.tolist())
        all_sparse.extend(sparse)

    return np.array(all_dense, dtype=np.float32), all_sparse


def build_faiss_index(dense_embeddings):
    """
    Build an inner-product FAISS index (cosine similarity on L2-normalized vectors).
    Uses IndexFlatIP — exact search, no quantization.
    """
    dim = dense_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(dense_embeddings)
    print(f"  FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index


def save_bge_m3_index(faiss_index, sparse_weights, doc_ids, texts, out_dir):
    """
    Persist BGE-M3 index artifacts to out_dir:
      - faiss.index     : FAISS IndexFlatIP
      - sparse.pkl      : list of sparse weight dicts
      - metadata.pkl    : doc_ids + texts
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(faiss_index, str(out_dir / "faiss.index"))

    with open(out_dir / "sparse.pkl", "wb") as f:
        pickle.dump(sparse_weights, f)

    with open(out_dir / "metadata.pkl", "wb") as f:
        pickle.dump({"doc_ids": doc_ids, "texts": texts}, f)

    total_size = sum(
        os.path.getsize(out_dir / fn) / 1e6
        for fn in ["faiss.index", "sparse.pkl", "metadata.pkl"]
    )
    print(f"  Saved to {out_dir}/ (total {total_size:.1f} MB)")

### 7a. Index HotpotQA Source Docs

In [ ]:
print("=" * 60)
print("BGE-M3 Indexing: HotpotQA source docs")
print("=" * 60)

t0 = time.time()
hotpotqa_dense, hotpotqa_sparse = embed_corpus_bge_m3(
    hotpotqa_texts, batch_size=32, desc="HotpotQA BGE-M3"
)
print(f"Embedding done in {(time.time()-t0)/60:.1f} min")
print(f"Dense shape: {hotpotqa_dense.shape}")
print(f"Sparse: {len(hotpotqa_sparse)} docs, sample keys: {list(hotpotqa_sparse[0].keys())[:5]}")

faiss_hotpotqa = build_faiss_index(hotpotqa_dense)

save_bge_m3_index(
    faiss_hotpotqa, hotpotqa_sparse, hotpotqa_ids, hotpotqa_texts,
    FAISS_HOTPOT
)

print("✓ HotpotQA BGE-M3 index saved.")

# Free memory before Wikipedia embedding
del hotpotqa_dense, hotpotqa_sparse
torch.cuda.empty_cache()
print("VRAM cleared.")

### 7b. Index Wikipedia ID Corpus

This is the heaviest step (~2–3 hours for 20K docs on T4). Checkpointing is done in batches of 2,000 docs — partial embeddings are saved to disk so we can resume if the session dies mid-way.

In [ ]:
CHECKPOINT_DIR  = WORKING_DIR / "wiki_embed_checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_SIZE = 2000  # docs per checkpoint batch

def embed_corpus_with_checkpoints(texts, ids, batch_size=32, checkpoint_size=2000, desc="Embedding"):
    """
    Embed corpus in chunks of `checkpoint_size` docs.
    Each chunk is saved to disk immediately.
    If a chunk file already exists (from a previous interrupted run), it is loaded
    instead of re-computed — allows resume without re-embedding.
    """
    all_dense  = []
    all_sparse = []
    n_chunks   = (len(texts) + checkpoint_size - 1) // checkpoint_size

    for chunk_idx in range(n_chunks):
        start = chunk_idx * checkpoint_size
        end   = min(start + checkpoint_size, len(texts))
        ckpt_path = CHECKPOINT_DIR / f"chunk_{chunk_idx:04d}.pkl"

        if ckpt_path.exists():
            print(f"  [Chunk {chunk_idx+1}/{n_chunks}] Loading from checkpoint...")
            with open(ckpt_path, "rb") as f:
                ckpt = pickle.load(f)
            all_dense.extend(ckpt["dense"])
            all_sparse.extend(ckpt["sparse"])
            continue

        print(f"  [Chunk {chunk_idx+1}/{n_chunks}] Embedding docs {start}–{end}...")
        chunk_texts  = texts[start:end]
        chunk_dense  = []
        chunk_sparse = []

        for i in tqdm(range(0, len(chunk_texts), batch_size),
                      desc=f"Chunk {chunk_idx+1}", leave=False):
            batch = chunk_texts[i : i + batch_size]
            output = bge_model.encode(
                batch,
                batch_size=batch_size,
                max_length=512,
                return_dense=True,
                return_sparse=True,
                return_colbert_vecs=False
            )
            dense  = output['dense_vecs']
            sparse = output['lexical_weights']

            norms = np.linalg.norm(dense, axis=1, keepdims=True)
            norms = np.where(norms == 0, 1.0, norms)
            dense = dense / norms

            chunk_dense.extend(dense.tolist())
            chunk_sparse.extend(sparse)

        # Persist chunk immediately
        with open(ckpt_path, "wb") as f:
            pickle.dump({"dense": chunk_dense, "sparse": chunk_sparse}, f)
        print(f"  ✓ Chunk {chunk_idx+1} saved ({ckpt_path.name})")

        all_dense.extend(chunk_dense)
        all_sparse.extend(chunk_sparse)

    return np.array(all_dense, dtype=np.float32), all_sparse


print("=" * 60)
print("BGE-M3 Indexing: Wikipedia ID corpus")
print(f"  Total docs: {len(wiki_texts)}")
print(f"  Checkpoint size: {CHECKPOINT_SIZE} docs")
print("=" * 60)

t0 = time.time()
wiki_dense, wiki_sparse = embed_corpus_with_checkpoints(
    wiki_texts, wiki_ids,
    batch_size=32,
    checkpoint_size=CHECKPOINT_SIZE,
    desc="Wikipedia ID BGE-M3"
)
elapsed = (time.time() - t0) / 60
print(f"\nTotal embedding time: {elapsed:.1f} min")
print(f"Dense shape: {wiki_dense.shape}")

In [ ]:
faiss_wiki = build_faiss_index(wiki_dense)

save_bge_m3_index(
    faiss_wiki, wiki_sparse, wiki_ids, wiki_texts,
    FAISS_WIKI
)

print("✓ Wikipedia ID BGE-M3 index saved.")

# Clean up checkpoint files — they've served their purpose
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
print("Checkpoint files cleaned up.")

del wiki_dense, wiki_sparse
torch.cuda.empty_cache()
print("VRAM cleared.")

## 8. RRF Fusion Utility (Sparse + Dense)

This cell defines the RRF fusion function used by NB06 and NB07 at query time. It is NOT run during indexing — included here so the logic lives in one place and can be imported.

RRF formula: `score(d) = Σ 1 / (k + rank(d))` where k=60.

In [ ]:
def rrf_fuse(dense_ranking, sparse_ranking, k=60):
    """
    Reciprocal Rank Fusion of dense and sparse rankings.

    Args:
        dense_ranking:  list of (doc_idx, score) sorted by dense score descending
        sparse_ranking: list of (doc_idx, score) sorted by sparse score descending
        k:              RRF smoothing constant (default 60)

    Returns:
        list of (doc_idx, rrf_score) sorted by RRF score descending
    """
    scores = {}

    for rank, (doc_idx, _) in enumerate(dense_ranking, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)

    for rank, (doc_idx, _) in enumerate(sparse_ranking, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def sparse_score_query(query_sparse_weights, doc_sparse_weights):
    """
    Compute BGE-M3 sparse similarity between a query and a document.
    Both are dicts of {token_id: weight}.
    Score = sum of min(q_w, d_w) for shared tokens.
    This matches the BGE-M3 paper's lexical matching formulation.
    """
    score = 0.0
    for token_id, q_weight in query_sparse_weights.items():
        if token_id in doc_sparse_weights:
            score += min(q_weight, doc_sparse_weights[token_id])
    return score


print("RRF fusion and sparse scoring utilities defined.")

# Quick sanity test
dense_ranks  = [(0, 0.9), (1, 0.8), (2, 0.6)]
sparse_ranks = [(1, 0.7), (0, 0.5), (3, 0.4)]
fused = rrf_fuse(dense_ranks, sparse_ranks)
print(f"RRF test (doc_idx, rrf_score): {fused}")
# Expected: doc 0 and 1 should both score higher than 2 or 3

## 9. Sanity Check: Cross-lingual Retrieval

Take 10 XQuAD-ID parallel pairs (5 EN, 5 ID versions of the same question).  
Query Wikipedia ID corpus with both BM25 and BGE-M3 Dense.  
Compare top-5 retrieved docs to document whether BM25 fails on EN→ID.

**This check empirically motivates the architecture and is cited in the paper.**

In [ ]:
print("Loading XQuAD parallel pairs for sanity check...")
xquad_pairs = load_jsonl(XQUAD_PARALLEL_PATH)
print(f"  Loaded {len(xquad_pairs)} XQuAD parallel pairs")

# Sample 5 pairs (deterministic seed)
import random
random.seed(42)
sample_pairs = random.sample(xquad_pairs, min(5, len(xquad_pairs)))

print(f"  Sampled {len(sample_pairs)} pairs for sanity check")
print()
for i, pair in enumerate(sample_pairs):
    q_en = pair.get('question_en', pair.get('question', ''))
    q_id = pair.get('question_id_text', pair.get('question_id', ''))
    print(f"  Pair {i+1}: EN='{q_en[:60]}...'")
    print(f"           ID='{q_id[:60]}...'")

In [ ]:
# ── Reload FAISS wiki index (in case VRAM was cleared above) ──────────────────
print("Reloading FAISS Wikipedia ID index for sanity check...")
faiss_wiki_check = faiss.read_index(str(FAISS_WIKI / "faiss.index"))
with open(FAISS_WIKI / "metadata.pkl", "rb") as f:
    wiki_meta = pickle.load(f)
wiki_ids_check   = wiki_meta["doc_ids"]
wiki_texts_check = wiki_meta["texts"]

# ── Reload BM25 wikipedia ─────────────────────────────────────────────────────
print("Reloading BM25 Wikipedia ID index...")
with open(INDEX_DIR / "bm25_wikipedia_id.pkl", "rb") as f:
    bm25_wiki_payload = pickle.load(f)
bm25_wiki_check = bm25_wiki_payload["bm25"]

print("Indexes loaded for sanity check.")

In [ ]:
def retrieve_bm25(query, bm25_obj, texts, top_k=5):
    """Retrieve top-k docs from BM25 index."""
    tokens = sastrawi_tokenize(query)
    if not tokens:
        return []
    scores = bm25_obj.get_scores(tokens)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx]), texts[idx][:80]) for idx in top_idxs]

def retrieve_bge_dense(query, faiss_index, texts, top_k=5):
    """Retrieve top-k docs from FAISS dense index."""
    output = bge_model.encode(
        [query],
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    q_vec = output['dense_vecs'][0:1].astype(np.float32)
    norm  = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    scores, idxs = faiss_index.search(q_vec, top_k)
    return [(int(idx), float(scores[0][i]), texts[idx][:80])
            for i, idx in enumerate(idxs[0]) if idx >= 0]


sanity_results = []

print("Running cross-lingual sanity check...\n")
for i, pair in enumerate(sample_pairs):
    q_en = pair.get('question_en', pair.get('question', ''))
    q_id = pair.get('question_id_text', pair.get('question_id', ''))
    answer = pair.get('answer', 'N/A')

    bm25_en   = retrieve_bm25(q_en,  bm25_wiki_check, wiki_texts_check)
    bm25_id   = retrieve_bm25(q_id,  bm25_wiki_check, wiki_texts_check)
    dense_en  = retrieve_bge_dense(q_en, faiss_wiki_check, wiki_texts_check)
    dense_id  = retrieve_bge_dense(q_id, faiss_wiki_check, wiki_texts_check)

    # Check whether the answer string appears in any top-5 retrieved doc
    def answer_hit(results, answer):
        return any(answer.lower() in text.lower() for _, _, text in results)

    result = {
        "pair_idx":    i,
        "question_en": q_en,
        "question_id": q_id,
        "answer":      answer,
        "bm25_en_top1":   bm25_en[0][2]  if bm25_en  else "",
        "bm25_id_top1":   bm25_id[0][2]  if bm25_id  else "",
        "dense_en_top1":  dense_en[0][2] if dense_en  else "",
        "dense_id_top1":  dense_id[0][2] if dense_id  else "",
        "bm25_en_score_top1":  bm25_en[0][1]  if bm25_en  else 0.0,
        "bm25_id_score_top1":  bm25_id[0][1]  if bm25_id  else 0.0,
        "dense_en_score_top1": dense_en[0][1] if dense_en  else 0.0,
        "dense_id_score_top1": dense_id[0][1] if dense_id  else 0.0,
    }
    sanity_results.append(result)

    print(f"Pair {i+1}: Q_EN = '{q_en[:60]}...'")
    print(f"  BM25  EN→ID top-1 score: {result['bm25_en_score_top1']:.3f} | '{result['bm25_en_top1']}'")
    print(f"  BM25  ID→ID top-1 score: {result['bm25_id_score_top1']:.3f} | '{result['bm25_id_top1']}'")
    print(f"  Dense EN→ID top-1 score: {result['dense_en_score_top1']:.4f} | '{result['dense_en_top1']}'")
    print(f"  Dense ID→ID top-1 score: {result['dense_id_score_top1']:.4f} | '{result['dense_id_top1']}'")
    print()

In [ ]:
# ── Aggregate sanity check stats ──────────────────────────────────────────────
avg_bm25_en  = np.mean([r['bm25_en_score_top1']  for r in sanity_results])
avg_bm25_id  = np.mean([r['bm25_id_score_top1']  for r in sanity_results])
avg_dense_en = np.mean([r['dense_en_score_top1'] for r in sanity_results])
avg_dense_id = np.mean([r['dense_id_score_top1'] for r in sanity_results])

summary = {
    "n_pairs": len(sanity_results),
    "avg_bm25_score_EN_query":   round(avg_bm25_en, 4),
    "avg_bm25_score_ID_query":   round(avg_bm25_id, 4),
    "bm25_gap_EN_minus_ID":      round(avg_bm25_en - avg_bm25_id, 4),
    "avg_dense_score_EN_query":  round(avg_dense_en, 4),
    "avg_dense_score_ID_query":  round(avg_dense_id, 4),
    "dense_gap_EN_minus_ID":     round(avg_dense_en - avg_dense_id, 4),
    "interpretation": (
        "bm25_gap > 0 means BM25 scores higher for EN queries — contradicts expected behavior "
        "(BM25 should favor ID queries on ID corpus). Negative bm25_gap = BM25 fails on EN. "
        "dense_gap near 0 = BGE-M3 is language-agnostic as expected."
    ),
    "per_pair_results": sanity_results
}

print("=" * 50)
print("SANITY CHECK SUMMARY")
print("=" * 50)
print(f"  BM25  avg score EN→ID corpus: {avg_bm25_en:.4f}")
print(f"  BM25  avg score ID→ID corpus: {avg_bm25_id:.4f}")
print(f"  BM25  gap (EN - ID):          {avg_bm25_en - avg_bm25_id:.4f}")
print()
print(f"  Dense avg score EN→ID corpus: {avg_dense_en:.4f}")
print(f"  Dense avg score ID→ID corpus: {avg_dense_id:.4f}")
print(f"  Dense gap (EN - ID):          {avg_dense_en - avg_dense_id:.4f}")
print()
print("  Expected: BM25 gap negative (BM25 fails on EN→ID)")
print("            Dense gap near 0 (BGE-M3 language-agnostic)")

# Save
sanity_path = RESULTS_DIR / "retrieval_sanity_check.json"
with open(sanity_path, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\n  Saved to {sanity_path}")

## 10. Verify All Artifacts Exist

In [ ]:
required_artifacts = [
    INDEX_DIR / "bm25_hotpotqa.pkl",
    INDEX_DIR / "bm25_wikipedia_id.pkl",
    FAISS_HOTPOT / "faiss.index",
    FAISS_HOTPOT / "sparse.pkl",
    FAISS_HOTPOT / "metadata.pkl",
    FAISS_WIKI   / "faiss.index",
    FAISS_WIKI   / "sparse.pkl",
    FAISS_WIKI   / "metadata.pkl",
    RESULTS_DIR  / "retrieval_sanity_check.json",
]

all_ok = True
print("Artifact verification:")
for path in required_artifacts:
    exists = path.exists()
    size   = path.stat().st_size / 1e6 if exists else 0
    status = f"✓ ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {status}  {path.relative_to(WORKING_DIR)}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All artifacts present. Ready to persist.")
else:
    print("⚠ Some artifacts are missing. Do NOT proceed to persist step.")

## 11. Persist to Kaggle Dataset (`crosslingual-rag-indexes`)

All artifacts are zipped by directory and staged to `/kaggle/working/`.  
After running this cell, go to **Notebook → Save & Run All** OR manually upload the zip files to the `crosslingual-rag-indexes` Kaggle Dataset.

**Do NOT skip this step.** FAISS indexes are ephemeral — they disappear when the session ends.

In [ ]:
print("Zipping indexes for persistent storage...")

# Zip BM25 pickles individually
for pkl_name in ["bm25_hotpotqa.pkl", "bm25_wikipedia_id.pkl"]:
    src  = INDEX_DIR / pkl_name
    dest = WORKING_DIR / pkl_name  # place at top-level for easy upload
    shutil.copy(src, dest)
    print(f"  Copied {pkl_name} ({dest.stat().st_size / 1e6:.1f} MB)")

# Zip FAISS directories
for faiss_dir, zip_name in [
    (FAISS_HOTPOT, "faiss_bge_m3_hotpotqa"),
    (FAISS_WIKI,   "faiss_bge_m3_wikipedia_id"),
]:
    zip_path = WORKING_DIR / zip_name
    shutil.make_archive(str(zip_path), "zip", str(faiss_dir.parent), faiss_dir.name)
    size_mb = (WORKING_DIR / f"{zip_name}.zip").stat().st_size / 1e6
    print(f"  Zipped {zip_name}.zip ({size_mb:.1f} MB)")

# Copy sanity check result
shutil.copy(
    RESULTS_DIR / "retrieval_sanity_check.json",
    WORKING_DIR / "retrieval_sanity_check.json"
)
print("  Copied retrieval_sanity_check.json")

print()
print("Files staged in /kaggle/working/:")
for f in sorted(WORKING_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

## 12. How to Load Indexes in Downstream Notebooks (NB04, NB06, NB07, NB08)

Copy this block to the top of any notebook that needs these indexes:

```python
import shutil, pickle, faiss
from pathlib import Path

IDX_INPUT = Path("/kaggle/input/crosslingual-rag-indexes")
FAISS_DIR = Path("/kaggle/working/faiss_db")
FAISS_DIR.mkdir(exist_ok=True)

# Unzip FAISS indexes
for zip_name, subdir in [
    ("faiss_bge_m3_hotpotqa.zip",     "faiss_bge_m3_hotpotqa"),
    ("faiss_bge_m3_wikipedia_id.zip", "faiss_bge_m3_wikipedia_id"),
]:
    shutil.unpack_archive(str(IDX_INPUT / zip_name), str(FAISS_DIR))

# Load FAISS
faiss_hotpotqa = faiss.read_index(str(FAISS_DIR / "faiss_bge_m3_hotpotqa" / "faiss.index"))
faiss_wiki     = faiss.read_index(str(FAISS_DIR / "faiss_bge_m3_wikipedia_id" / "faiss.index"))

# Load sparse weights
with open(FAISS_DIR / "faiss_bge_m3_hotpotqa" / "sparse.pkl", "rb") as f:
    hotpotqa_sparse = pickle.load(f)
with open(FAISS_DIR / "faiss_bge_m3_wikipedia_id" / "sparse.pkl", "rb") as f:
    wiki_sparse = pickle.load(f)

# Load metadata (doc_ids + texts)
with open(FAISS_DIR / "faiss_bge_m3_hotpotqa" / "metadata.pkl", "rb") as f:
    hotpotqa_meta = pickle.load(f)  # {"doc_ids": [...], "texts": [...]}
with open(FAISS_DIR / "faiss_bge_m3_wikipedia_id" / "metadata.pkl", "rb") as f:
    wiki_meta = pickle.load(f)

# Load BM25
with open(IDX_INPUT / "bm25_hotpotqa.pkl", "rb") as f:
    bm25_hotpotqa = pickle.load(f)["bm25"]
with open(IDX_INPUT / "bm25_wikipedia_id.pkl", "rb") as f:
    bm25_wiki = pickle.load(f)["bm25"]

print("All indexes loaded.")
```

## Done

| Artifact | Corpus | Used by |
|---|---|---|
| `bm25_hotpotqa.pkl` | HotpotQA source docs | NB06 (baseline ablation config) |
| `bm25_wikipedia_id.pkl` | Wikipedia ID | NB07 (Δgap BM25 baseline) |
| `faiss_bge_m3_hotpotqa/` | HotpotQA source docs | NB06 (ablation configs +Dense, +Hybrid, +BGE-M3 S+D, +Reranker, +CRAG) |
| `faiss_bge_m3_wikipedia_id/` | Wikipedia ID | NB04 (reranker), NB07 (Δgap), NB08 (RAGAS) |
| `retrieval_sanity_check.json` | — | Paper: qualitative evidence BM25 fails on EN→ID |

**Next step:** NB04 — Reranker setup and precision measurement.